In [1]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine

# 1. Paths
forest_path = os.path.join("..", "data", "Bird_Monitoring_Data_FOREST.XLSX")
grassland_path = os.path.join("..", "data", "Bird_Monitoring_Data_GRASSLAND.XLSX")
db_path = os.path.join("..", "data", "bird_monitoring.db")
csv_path = os.path.join("..", "data", "cleaned_bird_observations.csv")

# 2. Ingestion across all 11 sheets
def load_all_sheets(filepath, habitat_type):
    xls = pd.ExcelFile(filepath)
    dfs = []
    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet)
        df['Admin_Unit_Code'] = sheet
        df['Location_Type'] = habitat_type
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

df_forest = load_all_sheets(forest_path, "Forest")
df_grassland = load_all_sheets(grassland_path, "Grassland")

# Standardize Taxon column names
df_forest.rename(columns={'NPSTaxonCode': 'Taxon_Code'}, inplace=True)
df_grassland.rename(columns={'TaxonCode': 'Taxon_Code'}, inplace=True)

df_all = pd.concat([df_forest, df_grassland], ignore_index=True)

# 3. Feature Engineering & Cleaning
df_all['Date'] = pd.to_datetime(df_all['Date'])
df_all['Year'] = df_all['Date'].dt.year
df_all['Month'] = df_all['Date'].dt.month
df_all['Month_Name'] = df_all['Date'].dt.month_name()

def get_season(month):
    if month in [3, 4, 5]: return "Spring"
    elif month in [6, 7, 8]: return "Summer"
    elif month in [9, 10, 11]: return "Fall"
    else: return "Winter"

df_all['Season'] = df_all['Month'].apply(get_season)
df_all['Observation_Hour'] = df_all['Start_Time'].astype(str).str.extract(r'(\d{1,2}):')[0].astype(float)

# Impute Categorical & Boolean Fields
df_all['Sex'] = df_all['Sex'].fillna('Undetermined').replace({'': 'Undetermined'})
df_all['Distance'] = df_all['Distance'].fillna('Unknown')
df_all['ID_Method'] = df_all['ID_Method'].fillna('Unknown')
df_all['Sub_Unit_Code'] = df_all['Sub_Unit_Code'].fillna('None')
df_all['Site_Name'] = df_all['Site_Name'].fillna('None')
df_all['Previously_Obs'] = df_all['Previously_Obs'].fillna(False).astype(bool)

for col in ['Flyover_Observed', 'PIF_Watchlist_Status', 'Regional_Stewardship_Status', 'Initial_Three_Min_Cnt']:
    df_all[col] = df_all[col].astype(bool)

df_all['Temperature'] = pd.to_numeric(df_all['Temperature'], errors='coerce')
df_all['Humidity'] = pd.to_numeric(df_all['Humidity'], errors='coerce')

# Deduplication
df_all.drop_duplicates(inplace=True)

# 4. Save to SQLite and CSV
engine = create_engine(f"sqlite:///{db_path}")
df_all.to_sql('bird_observations', con=engine, if_exists='replace', index=False)
df_all.to_csv(csv_path, index=False)

print(f"Data Cleaning Completed! Total Valid Records: {len(df_all)}")

Data Cleaning Completed! Total Valid Records: 15368
